**Text-To-SQL Engine**

In [2]:
#schema context provided by Schema Extraction Module
schema="""

Table: credit_card_transactions
Columns:
- credit_transaction_id: bigint
- customer_id: bigint
- transaction_date: timestamp without time zone
- amount: numeric
- merchant_name: character varying
- card_network: character varying
- card_type: character varying
- transaction_status: character varying
- credit_limit: numeric

Table: customers
Columns:
- customer_id: bigint
- full_name: character varying
- email: character varying
- phone: character varying
- city: character varying
- state: character varying
- registration_date: date

Table: debit_card_transactions
Columns:
- debit_transaction_id: bigint
- customer_id: bigint
- transaction_date: timestamp without time zone
- amount: numeric
- merchant_name: character varying
- card_network: character varying
- account_type: character varying
- transaction_status: character varying
- transaction_type: character varying

Table: upi_transactions
Columns:
- upi_transaction_id: bigint
- customer_id: bigint
- transaction_date: timestamp without time zone
- amount: numeric
- merchant_name: character varying
- upi_app: character varying
- transaction_status: character varying
- transaction_type: character varying


"""

In [11]:
user_query="Select Top 10 Spending Customers In upi_transactions Table And Order Them With Respect To customer_id, the ouput should contain customer's name, customer id and total amount spent."

In [ ]:
!pip install langchain_huggingface

In [12]:
HUGGINGFACEHUB_API_TOKEN=""

In [13]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id="XGenerationLab/XiYanSQL-QwenCoder-7B-2504",
    task="text-generation",
    provider="featherless-ai",
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
    max_new_tokens=512,
    temperature=0.1,   # low temp — you want deterministic SQL, not creative variation
)

model = ChatHuggingFace(llm=llm)

prompt1 = f"""
You are an expert SQL developer. Convert the user's natural language question into a single, correct SQL query based only on the database schema provided below.

### Database Schema
{schema}

### Rules
1. Only use tables and columns that exist in the schema above. Never invent column or table names.
2. Use explicit JOINs (not comma joins), and qualify column names with table names/aliases when more than one table is involved.
3. Use the exact SQL dialect: PostgreSQL.
4. If the question is ambiguous or cannot be answered from the schema, respond with: -- CANNOT_ANSWER: <short reason>
5. Do not include explanations, comments, or markdown formatting — output only the raw SQL query.
6. Prefer readable formatting: one clause per line (SELECT, FROM, WHERE, GROUP BY, ORDER BY).
7. Use LIMIT only if the user asks for "top N" / "first N" results.

### User Question
{user_query}

### SQL Query
"""

try:
    result1 = model.invoke(prompt1)
    sql_query = result1.content
except Exception as e:
    print(f"Inference call failed: {e}")
    sql_query = None

In [14]:
sql_query

'SELECT c.full_name AS customer_name, u.customer_id, SUM(u.amount) AS total_amount_spent FROM upi_transactions u JOIN customers c ON u.customer_id = c.customer_id GROUP BY c.full_name, u.customer_id ORDER BY u.customer_id, total_amount_spent DESC LIMIT 10;\n'